# Data preparation for Llama
Stage 2 & 3

## A Overview

In [1]:
from pathlib import Path
import pandas as pd

from datasets import Dataset
from transformers import BertForSequenceClassification, BertTokenizer

import configuration

from src import hf_utils, setup
from src.models import bert

/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bert_model_path = Path("..") / "models" / "BERT"/ "w1_o0.735494" / "1.0"
tokens_path = Path("../tokens/BERT/llama_pre_bert_set")
datasets_path = Path("..") / "data"/"splitted"

device = setup.setup_device_with_seeds()

Using device: mps


In [3]:
bert_model = BertForSequenceClassification.from_pretrained(bert_model_path)
bert_tokenizer = BertTokenizer.from_pretrained(bert_model_path)
bert_model.to(device)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8801.54it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
df_pre_bert = pd.read_csv(datasets_path / "llama_pre_bert_sets.csv")

# For testing purposes, we can sample a smaller subset of the data
df_pre_bert = df_pre_bert.sample(n=100, random_state=42).reset_index(drop=True)

/var/folders/yh/9qq7z2f14f14bdj_1k2673700000gn/T/ipykernel_97918/1635665263.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pre_bert = pd.read_csv(datasets_path / "llama_pre_bert_set.csv")


In [ ]:
if tokens_path.exists():
    print("Loading tokenized datasets from disk...")
    pre_bert_tokenized = Dataset.load_from_disk(tokens_path)
else:
    
    ds_pre_bert = Dataset.from_pandas(df_pre_bert)
    
    # Check max token length for the 'tweet_text' column
    hf_utils.max_length_dist(df_pre_bert, 'tweet_text', bert_tokenizer)
    
    pre_bert_tokenized = hf_utils.tokenize(ds_pre_bert, bert_tokenizer, tokens_path, bert.format_dataset)

Loading tokenized datasets from disk...


In [6]:
predictions, confidences = bert.predict(bert_model, pre_bert_tokenized, device, confidence_scores=True)
bert.report_metrics(pre_bert_tokenized, predictions)

Predicting:: 100%|██████████| 7/7 [00:01<00:00,  5.37it/s]


Classification report:
              precision    recall  f1-score   support

       False     0.9500    1.0000    0.9744        95
        True     0.0000    0.0000    0.0000         5

    accuracy                         0.9500       100
   macro avg     0.4750    0.5000    0.4872       100
weighted avg     0.9025    0.9500    0.9256       100




/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _wa

In [7]:
df_pre_bert["predicted"] = predictions
df_pre_bert["confidence"] = confidences
df_pre_bert.head()

,uid,tweet_text,informative,humanitarian_label,subset,predicted,confidence
0,25888,Â« storm You're like the wind I'm like the ra...,False,NaN,disaster,0,0.995108
1,6760798,@barakobama naaah we straight!!,False,NaN,out_topic,0,0.995470
2,7917229,Aw the guys r cooking while the girls play vid...,False,NaN,out_topic,0,0.995407
3,4316004,Agassi iconfiesai su iinfelicidadi en autobiog...,False,NaN,out_topic,0,0.995108
4,1919491,"@perpinya Oh yep, wow, I definitely felt that!...",False,NaN,out_topic,0,0.995203


In [8]:
df_failed_predictions = df_pre_bert[df_pre_bert["informative"] != df_pre_bert["predicted"]]
df_failed_predictions.head()

,uid,tweet_text,informative,humanitarian_label,subset,predicted,confidence
11,88132,cried a bit when i read and watched stuff abou...,True,NaN,disaster,0,0.995097
29,203246,@vausecnn the rain is drifting away from SE au...,True,NaN,disaster,0,0.995120
78,76699,Severe drought followed by floods have slashed...,True,other_relevant_information,disaster,0,0.995227
81,74202,.@narendramodi declining UAE monetary aid for ...,True,rescue_volunteering_or_donation_effort,disaster,0,0.994952
87,28172,RT @CCOT_MAGA: MSM and #SanJuanMayor blame @PO...,True,other_relevant_information,disaster,0,0.995112


In [9]:
threshold = 0.2
df_low_confidence = df_pre_bert[df_pre_bert["confidence"] < threshold]
df_low_confidence.head()

,uid,tweet_text,informative,humanitarian_label,subset,predicted,confidence


In [10]:
df = pd.concat([df_failed_predictions, df_low_confidence]).drop_duplicates().reset_index(drop=True)
df.to_csv(datasets_path / "llama_set.csv", index=False)